In [30]:
# Optional when there's an incompatibility with bitsandbytes - Reinstall from GitHub so it builds for CUDA 12.5
#%pip install --no-cache-dir git+https://github.com/TimDettmers/bitsandbytes.git@main

  Cloning https://github.com/TimDettmers/bitsandbytes.git (to revision main) to /tmp/pip-req-build-h9yc3ksv
  Running command git clone --filter=blob:none --quiet https://github.com/TimDettmers/bitsandbytes.git /tmp/pip-req-build-h9yc3ksv
  Resolved https://github.com/TimDettmers/bitsandbytes.git to commit 97073cdb8a78618b8a56f51a9495254b645fd085
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for bitsandbytes: filename=bitsandbytes-0.46.0.dev0-cp311-cp311-linux_x86_64.whl size=92265 sha256=f51ac252951f352924a8ad8498722bca992e47bd916d58417ad4488b74c48e03
  Stored in directory: /tmp/pip-ephem-wheel-cache-0bkx_l0o/wheels/10/45/eb/d737947ef61806b694f1607f5e448af71543969b11530b261b
Successfully built bitsandbytes


In [2]:
 #%% Importing libraries
import os
from huggingface_hub import login
import transformers
import torch
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model
)

from datasets import Dataset

ERROR:bitsandbytes.cextension:Could not load bitsandbytes native library: /usr/local/lib/python3.11/dist-packages/bitsandbytes/libbitsandbytes_cpu.so: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/bitsandbytes/cextension.py", line 87, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/bitsandbytes/cextension.py", line 74, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/ctypes/__init__.py", line 454, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.11/dist-packages/bitsandbytes/libbitsandbytes_cpu.so: cannot op

In [3]:
import bitsandbytes as bnb
import importlib
importlib.reload(bnb)
from bitsandbytes.nn import Linear8bitLt


In [4]:
# Loading HuggingFace Token for Colab
login(token='') 

In [6]:
#%% Load and prepare data
normalized_product_attributes = pd.read_excel('../Data/Normalized_product_attribute_name.xlsx', sheet_name='Normalized Product Attributes')

In [7]:
dataset = pd.read_csv('../Data/Product_Normalization_GRI_Expanded.csv')

In [12]:


def create_stratified_sample(dataset, n_samples_per_class, seed = 42):
    """
    Creates a stratified sample from dataset
    """

    # Seed everything
    random.seed(seed)
    np.random.seed(seed)

    # Rest of the sampling process
    room_types = dataset['Guest Room Info'].unique()
    sample_rows = []
    sampled_indices = set()

    print(f"\nSampling {n_samples_per_class} descriptions per room type:")
    print("=" * 50)

    for room_type in room_types:
        room_type_data = dataset[dataset['Guest Room Info'] == room_type]
        available = len(room_type_data)

        if available < n_samples_per_class:
            print(f"\nWarning: Only {available} samples available for {room_type}")
            n_to_sample = available
        else:
            n_to_sample = n_samples_per_class

        sampled = room_type_data.sample(n=n_to_sample, random_state=seed)
        sample_rows.append(sampled)
        sampled_indices.update(sampled.index)

        print(f"\n{room_type}:")
        print(f"- Sampled {n_to_sample} from {available} available")

    sample_df = pd.concat(sample_rows, axis=0).reset_index(drop=True)
    remaining_df = dataset[~dataset.index.isin(sampled_indices)].reset_index(drop=True)

    print(f"\nFinal counts:")
    print(f"Sample size: {len(sample_df)}")
    print(f"Remaining size: {len(remaining_df)}")

    return sample_df, remaining_df


In [13]:
# 1% of each room type as training data

n_samples_per_class = 10
sample_df_1pct, remaining_df_1pct = create_stratified_sample(dataset, n_samples_per_class)


Sampling 10 descriptions per room type:

Accessible Room:
- Sampled 10 from 962 available

Suite:
- Sampled 10 from 987 available

Executive/Club Suite:
- Sampled 10 from 860 available

Double Bed:
- Sampled 10 from 815 available

King Bedroom:
- Sampled 10 from 921 available

Queen Bedroom:
- Sampled 10 from 813 available

Penthouse:
- Sampled 10 from 975 available

Studio Suite:
- Sampled 10 from 864 available

Twin Room:
- Sampled 10 from 738 available

Family Room/Suite:
- Sampled 10 from 965 available

Cottage:
- Sampled 10 from 1000 available

Loft:
- Sampled 10 from 963 available

Guest Room:
- Sampled 10 from 694 available

Bungalow:
- Sampled 10 from 978 available

Villa:
- Sampled 10 from 927 available

Junior Suite:
- Sampled 10 from 922 available

Executive/Club Room:
- Sampled 10 from 981 available

Classic Room:
- Sampled 10 from 907 available

Comfort Room:
- Sampled 10 from 900 available

Deluxe Room:
- Sampled 10 from 955 available

Deluxe Suite:
- Sampled 10 from 954

In [14]:
labels = sample_df_1pct['Guest Room Info'].unique()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(labels)

print(f"Number of labels: {num_labels}")
model_name = "meta-llama/Llama-3.2-1B-Instruct"
output_dir = "llama3.2_1b"



Number of labels: 35


In [21]:
# Training script:

def train_lora_fp32(
    full_sample_data,  # pandas.DataFrame with "Room Description","Guest Room Info"
    label2id,
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    output_dir="lora_fp32_clean"
):
    # 1) Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.pad_token is None:
        # Add a real pad token so we can batch>1
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    # 2) Load the full‑precision model onto GPU (if available)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label={v:k for k,v in label2id.items()},
        label2id=label2id,
    ).to(device)
    # Tell the model its pad_token_id, then resize embeddings
    model.config.pad_token_id = tokenizer.pad_token_id
    model.resize_token_embeddings(len(tokenizer))


    # 3) Enable gradient checkpointing and attach LoRA adapters
    model.gradient_checkpointing_enable()
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["q_proj","v_proj"],
        inference_mode=False,
        bias="none"
    )
    model = get_peft_model(model, lora_cfg)

    # 4) Build a HuggingFace Dataset
    df = full_sample_data.rename(
        columns={"Room Description Expanded":"text","Guest Room Info":"label"}
    )[["text","label"]]
    ds = Dataset.from_pandas(df)

    def preprocess(examples):
        enc = tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=128
        )
        enc["labels"] = [label2id[l] for l in examples["label"]]
        return enc

    tokenized = ds.map(
        preprocess, batched=True,
        remove_columns=["text","label"]
    )

    # 5) Trainer
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=3e-4,
        fp16=(device=="cuda"),     # use fp16 on GPU if available
        bf16=False,
        logging_steps=20,
        save_strategy="epoch",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized,
        tokenizer=tokenizer,
    )

    # 6) Train & save
    trainer.train()
    model.save_pretrained(output_dir)    # only LoRA adapters
    tokenizer.save_pretrained(output_dir)
    return model, tokenizer



In [28]:
torch.cuda.is_available()

True

In [22]:
model, tokenizer = train_lora_fp32(
    full_sample_data=sample_df_1pct,
    label2id=label2id
)


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/350 [00:00<?, ? examples/s]

<ipython-input-21-f06c95e751ce>:87: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
20,4.534600
40,3.557800
60,2.886700


In [24]:

from sklearn.metrics import accuracy_score, classification_report

def classify_remaining(
    model,
    tokenizer,
    label2id,
    remaining_df,
    output_file="preds_with_metrics.csv"
):
    """
    - remaining_df must have columns:
        * "Room Description Expanded"
        * "Guest Room Info"
    - Returns a DataFrame with:
        * text, actual label, predicted label, confidence
    - Prints overall accuracy and a classification report.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    # invert label2id
    id2label = {v: k for k, v in label2id.items()}

    texts  = remaining_df["Room Description Expanded"].tolist()
    actual = remaining_df["Guest Room Info"].tolist()

    preds = []
    confs = []

    with torch.no_grad():
        for text in tqdm(texts, desc="Classifying"):
            enc = tokenizer(
                text,
                truncation=True,
                padding="max_length",
                max_length=128,
                return_tensors="pt"
            ).to(device)

            logits = model(**enc).logits
            probs  = torch.softmax(logits, dim=-1)[0]
            pid    = int(probs.argmax().cpu().item())
            conf   = float(probs[pid].cpu().item())

            preds.append(id2label[pid])
            confs.append(conf)

    # Build results DataFrame
    results_df = pd.DataFrame({
        "Room Description Expanded": texts,
        "Actual_Room_Type":            actual,
        "Predicted_Room_Type":         preds,
        "Confidence":                  confs
    })

    # Compute and print metrics
    acc = accuracy_score(actual, preds)
    print(f"\nOverall Accuracy: {acc:.2%}\n")
    print("Classification Report:")
    print(classification_report(actual, preds, zero_division=0))

    # Save to CSV
    results_df.to_csv(output_file, index=False)
    return results_df


In [25]:
results_df = classify_remaining(
    model,
    tokenizer,
    label2id,
    remaining_df_1pct,
    output_file="predictions_and_metrics.csv"
)
#9.49%

Classifying:   0%|          | 0/30887 [00:00<?, ?it/s]


Overall Accuracy: 9.49%

Classification Report:
                      precision    recall  f1-score   support

     Accessible Room       0.06      0.02      0.03       952
           Apartment       0.15      0.11      0.13       949
            Bungalow       0.07      0.04      0.05       968
              Cabana       0.19      0.25      0.22       974
        Classic Room       0.07      0.03      0.05       897
       Classic Suite       0.10      0.14      0.11       836
        Comfort Room       0.07      0.13      0.09       890
             Cottage       0.16      0.19      0.17       990
         Deluxe Room       0.05      0.12      0.07       945
        Deluxe Suite       0.08      0.09      0.08       944
          Double Bed       0.21      0.14      0.16       805
 Executive/Club Room       0.03      0.01      0.01       971
Executive/Club Suite       0.05      0.03      0.04       850
   Family Room/Suite       0.05      0.05      0.05       955
          Guest Room

In [ ]:
results_df

In [23]:
remaining_df_1pct

,Unnamed: 0,Room Description,Guest Room Info,Room Description Expanded
0,0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...
...,...,...,...,...
30882,31232,BEST FLEXIBLE RATE|PREMIUM OCEAN VIEW ROOM WHE...,Run of the House,BEST FLEXIBLE RATE|PREMIUM OCEAN VIEW ROOM WHE...
30883,31233,1000 BONUS POINTS NT INCLUDES|ROOM AND 1000 RE...,Run of the House,1000 BONUS POINTS NT INCLUDES|ROOM AND 1000 RE...
30884,31234,BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WH...,Run of the House,BEST FLEXIBLE RATE|HOLIDAY INN EXPRESS ROOM WH...
30885,31235,GREATRATE DISCOUNTED STAYS. 2|RUN OF HOUSE STA...,Run of the House,GREATRATE DISCOUNTED STAYS. 2|RUN OF HOUSE STA...
